# Cross-Validation Predictive Models With GraphEstimator + Graph Vectorizers

Thin notebook for fitting classical graph predictive models on persisted PCNs with 5-fold stratified grouped cross-validation. It uses `abstractgraph_ml.estimators.GraphEstimator` with a selectable graph vectorizer such as `SimplexPairGraphVectorizer` or `NSPPK`.


## 1. Configure

In [2]:
try:
    from _bootstrap import configure_paths
except ImportError:
    from notebooks._bootstrap import configure_paths

PROJECT_ROOT = configure_paths(include=("abstractgraph_graphicalizer", "simplex_pair_vectorizer", "abstractgraph_ml"))

from datetime import datetime, timezone

from enzyme_classification.modeling.datasets import load_graph_dataset_from_status
from enzyme_classification.modeling.results import persist_graph_estimator_results
from enzyme_classification.modeling.splits import make_stratified_grouped_cross_validation_splits
from abstractgraph_graphicalizer.protein import ProteinLabelGraphicalizer
from abstractgraph_ml.estimators import GraphEstimator
from simplex_pair_vectorizer import SimplexPairGraphVectorizer

try:
    from nsppk import NSPPK
except ImportError:
    NSPPK = None

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)


### Node Label Alphabet Options

`NODE_LABEL_ALPHABET` controls how amino-acid node labels are grouped before graph vectorization. The graph topology is unchanged; only the node label seen by the vectorizer changes.

- `aa20`: keeps the original 20 amino-acid residue labels. This is the most detailed setting and is the baseline for comparison.
- `hp2`: reduces residues to hydrophobic or polar. This tests whether very coarse chemistry is enough.
- `charge4`: separates positive, negative, neutral polar, and nonpolar residues. This emphasizes electrostatic and polarity differences.
- `physchem5`: groups residues into hydrophobic, polar uncharged, positive, negative, and special residues. This is a compact general-purpose physicochemical alphabet.
- `chem7`: uses more specific side-chain chemistry: acidic, basic, hydroxyl, amide, sulfur, aromatic, and aliphatic/special. This keeps enzyme-relevant chemistry more visible while still reducing sparsity.
- `dayhoff6`: groups residues by a classic substitution-style reduced alphabet. This emphasizes evolutionary exchangeability rather than hand-picked chemistry.


In [3]:

TASK = "B"  # "A" for enzyme vs non-enzyme, "B" for first-level EC class
RANDOM_STATE = 4
CV_N_SPLITS = 5
MIN_SAMPLES_PER_CLASS_FOR_EVALUATION = 5
FILTER_UNDER_SUPPORTED_CLASSES = False
SPLIT_GROUP_COLUMN = "uniprot_accessions"  # Keep all structures for the same UniProt protein sequence in one fold side.
SPLIT_GROUP_FALLBACK_COLUMNS = ["protein_name", "entity_id", "pdb_id"]
TRANSFORMER_KIND = "simplex_pair"  # "simplex_pair" or "nsppk"
NODE_LABEL_ALPHABET = "aa20"  # "aa20", "hp2", "charge4", "physchem5", "chem7", or "dayhoff6"
EDGE_DISTANCE_THRESHOLDS = None  # None => "contact"; e.g. [5.0] => "contact_close"/"contact_far".
BASE_ESTIMATOR_KIND = "linear_sgd"  # "linear_sgd" or "random_forest"


TASK_A_STATUS = PROJECT_ROOT / "data/processed/task_a/graph_build_status.tsv"
TASK_B_STATUS = PROJECT_ROOT / "data/processed/task_b/graph_build_status.tsv"

STATUS_PATH = TASK_A_STATUS if TASK.upper() == "A" else TASK_B_STATUS
RESULTS_ROOT = PROJECT_ROOT / "results" / f"task_{TASK.lower()}" / "cross_validation"
STATUS_PATH


PosixPath('/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/task_b/graph_build_status.tsv')

## 2. Load Persisted Graph Dataset

In [4]:
dataset = load_graph_dataset_from_status(STATUS_PATH, task=TASK)
graphs = dataset.graphs
y = dataset.labels
metadata = dataset.metadata

print(f"task={TASK}")
print(f"status_path={STATUS_PATH}")
print(f"graphs={len(graphs)} labels={len(y)}")
metadata[["sample_id", "pdb_id", "protein_name", "first_level_ec_numbers", "target", "graph_path"]].head()

task=B
status_path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/task_b/graph_build_status.tsv
graphs=66581 labels=66581


,sample_id,pdb_id,protein_name,first_level_ec_numbers,target,graph_path
0,1f0s.A,1f0s,COAGULATION FACTOR XA,3,3,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
1,1f0s.B,1f0s,COAGULATION FACTOR XA,3,3,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
2,1f5q.A,1f5q,CYCLIN DEPENDENT KINASE 2,2,2,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
3,1f5q.C,1f5q,CYCLIN DEPENDENT KINASE 2,2,2,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...
4,1f8u.A,1f8u,ACETYLCHOLINESTERASE,3,3,/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b6...


In [5]:
pd.Series(y, name="target").value_counts().sort_index()

target
1     9753
2    23731
3    28103
4     2337
5     2050
6      607
Name: count, dtype: int64

## 3. Cross-Validation Splits


In [6]:
cv_result = make_stratified_grouped_cross_validation_splits(
    graphs,
    y,
    metadata,
    n_splits=CV_N_SPLITS,
    random_state=RANDOM_STATE,
    min_samples_per_class_for_evaluation=MIN_SAMPLES_PER_CLASS_FOR_EVALUATION,
    filter_under_supported_classes=FILTER_UNDER_SUPPORTED_CLASSES,
    split_group_column=SPLIT_GROUP_COLUMN,
    split_group_fallback_columns=SPLIT_GROUP_FALLBACK_COLUMNS,
)

graphs_for_split = cv_result.graphs_for_split
y_for_split = cv_result.y_for_split
metadata_for_split = cv_result.metadata_for_split
all_labels = cv_result.all_labels
class_counts = cv_result.class_counts
under_supported_classes = cv_result.under_supported_classes
split_counts = cv_result.split_counts
active_under_supported_classes = cv_result.active_under_supported_classes
group_counts = cv_result.group_counts
class_group_counts = cv_result.class_group_counts
folds = cv_result.folds
split_strategy = cv_result.split_strategy
stratified = cv_result.stratified

print("class support")
print(class_counts)
print("class group support")
print(class_group_counts)

if not under_supported_classes.empty:
    print("Smoke-test warning: these classes have fewer than "
        f"{MIN_SAMPLES_PER_CLASS_FOR_EVALUATION} samples and cannot support a reliable held-out evaluation."
    )
    print(under_supported_classes)
if FILTER_UNDER_SUPPORTED_CLASSES and not under_supported_classes.empty:
    print(f"Filtered to classes with >= {MIN_SAMPLES_PER_CLASS_FOR_EVALUATION} samples: {all_labels}")

print(f"folds={len(folds)} strategy={split_strategy} stratified={stratified} "
    f"group_column={SPLIT_GROUP_COLUMN}"
)
print(f"groups total={group_counts.size}")
print("largest groups")
print(group_counts.head(10))

fold_summary_rows = []
for fold in folds:
    fold_summary_rows.append(
        {
            "fold_id": fold.fold_id,
            "n_train": len(fold.X_train),
            "n_test": len(fold.X_test),
            "n_train_groups": len(fold.train_groups),
            "n_test_groups": len(fold.test_groups),
            "group_overlap": len(fold.group_overlap),
            "missing_from_train": ";".join(map(str, fold.missing_from_train)),
            "missing_from_test": ";".join(map(str, fold.missing_from_test)),
            "valid_held_out_evaluation": fold.valid_held_out_evaluation,
        }
    )
fold_summary = pd.DataFrame(fold_summary_rows)
fold_summary


class support
1     9753
2    23731
3    28103
4     2337
5     2050
6      607
Name: count, dtype: int64
class group support
label
1     330
2    1123
3    1158
4      74
5      87
6      65
Name: split_group, dtype: int64
folds=5 strategy=stratified_group_kfold_5 stratified=True group_column=uniprot_accessions
groups total=2834
largest groups
uniprot_accessions:q9y233    1170
uniprot_accessions:p02794    1163
uniprot_accessions:p00918    1107
uniprot_accessions:p00734    1030
uniprot_accessions:p56817     712
uniprot_accessions:p00441     593
uniprot_accessions:p55072     574
uniprot_accessions:p24941     570
uniprot_accessions:p01116     562
uniprot_accessions:p00533     527
Name: count, dtype: int64


,fold_id,n_train,n_test,n_train_groups,n_test_groups,group_overlap,missing_from_train,missing_from_test,valid_held_out_evaluation
0,0,53424,13157,2264,570,0,,,True
1,1,53425,13156,2269,565,0,,,True
2,2,52627,13954,2275,559,0,,,True
3,3,53424,13157,2265,569,0,,,True
4,4,53424,13157,2263,571,0,,,True


## 4. Define GraphEstimator Transformer


In [7]:
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import Normalizer


def make_model(random_state: int) -> GraphEstimator:
    if TRANSFORMER_KIND == "nsppk":
        if NSPPK is None:
            raise ImportError("TRANSFORMER_KIND='nsppk' requires the nsppk package on PYTHONPATH.")
        graph_vectorizer = NSPPK(
            radius=1,
            distance=4,
            connector=1,
            nbits=14,
            dense=False,
            parallel=True,
        )
    elif TRANSFORMER_KIND == "simplex_pair":
        graph_vectorizer = SimplexPairGraphVectorizer(
            max_simplex_dimension=3,
            min_distance=0,
            max_distance=4,
            distance_mode="representative",
            distance_jitter="auto",
            pairing_mode="rooted",
            nbits=14,
            dense=False,
            include_self=True,
            cache_graphs=False,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unknown TRANSFORMER_KIND={TRANSFORMER_KIND!r}")

    transformer = Pipeline(
        [
            (
                "protein_label_graphicalizer",
                ProteinLabelGraphicalizer(
                    alphabet=NODE_LABEL_ALPHABET,
                    node_label_key="label",
                    edge_label_key="label",
                    edge_distance_key="distance",
                    edge_distance_thresholds=EDGE_DISTANCE_THRESHOLDS,
                ),
            ),
            ("graph_vectorizer", graph_vectorizer),
        ]
    )

    if BASE_ESTIMATOR_KIND == "linear_sgd":
        base_estimator = make_pipeline(
            Normalizer(norm="l2"),
            SGDClassifier(
                loss="hinge",
                alpha=1e-4,
                max_iter=5000,
                tol=1e-4,
                n_iter_no_change=20,
                random_state=random_state,
                class_weight="balanced",
                average=True,
                n_jobs=-1,
            ),
        )
    elif BASE_ESTIMATOR_KIND == "random_forest":
        base_estimator = RandomForestClassifier(
            n_estimators=1000,
            max_features="sqrt",
            min_samples_leaf=2,
            random_state=random_state,
            class_weight="balanced_subsample",
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unknown BASE_ESTIMATOR_KIND={BASE_ESTIMATOR_KIND!r}")

    return GraphEstimator(
        transformer=transformer,
        estimator=base_estimator,
        manifold=None,
        n_selected_features=None,
    )


model = make_model(RANDOM_STATE)
model


,transformer,"Pipeline(step..., nbits=14))])"
,estimator,Pipeline(step...tol=0.0001))])
,manifold,<abstractgrap...x77bef8506a50>
,n_selected_features,None
,"norm norm: {'l1', 'l2', 'max'}, default='l2'The norm to use to normalize each non zero sample. If norm='max'is used, values will be rescaled by the maximum of the absolutevalues.",'l2'
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array or a scipy.sparseCSR matrix).",True
,"loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'hinge'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True


## 5. Fit Cross-Validation Folds


In [ ]:
cv_run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
fold_results = []

for fold in folds:
    print(f"Fitting fold {fold.fold_id}: train={len(fold.X_train)} test={len(fold.X_test)}")
    fold_model = make_model(RANDOM_STATE + fold.fold_id)
    fold_model.fit(fold.X_train, fold.y_train)
    y_pred = fold_model.predict(fold.X_test)

    report = classification_report(
        fold.y_test,
        y_pred,
        labels=all_labels,
        output_dict=True,
        zero_division=0,
    )
    raw_report_df = pd.DataFrame(report).T
    report_df = raw_report_df.loc[
        [label for label in all_labels if label in raw_report_df.index],
        ["precision", "recall", "f1-score", "support"],
    ].copy()

    summary_rows = [
        {"metric": "accuracy", "value": accuracy_score(fold.y_test, y_pred), "support": len(fold.y_test)},
        {
            "metric": "balanced_accuracy",
            "value": balanced_accuracy_score(fold.y_test, y_pred),
            "support": len(fold.y_test),
        },
        {
            "metric": "adjusted_balanced_accuracy",
            "value": balanced_accuracy_score(fold.y_test, y_pred, adjusted=True),
            "support": len(fold.y_test),
        },
    ]
    for average_name, prefix in [("macro avg", "macro"), ("weighted avg", "weighted")]:
        average_report = report[average_name]
        summary_rows.extend(
            [
                {"metric": f"{prefix}_precision", "value": average_report["precision"], "support": average_report["support"]},
                {"metric": f"{prefix}_recall", "value": average_report["recall"], "support": average_report["support"]},
                {"metric": f"{prefix}_f1", "value": average_report["f1-score"], "support": average_report["support"]},
            ]
        )
    summary_report_df = pd.DataFrame(summary_rows).set_index("metric")
    metrics_df = summary_report_df[["value"]].copy()
    metrics_df.loc["valid_held_out_evaluation", "value"] = fold.valid_held_out_evaluation

    cm = confusion_matrix(fold.y_test, y_pred, labels=all_labels)
    confusion_matrix_df = pd.DataFrame(
        cm,
        index=[f"true_{label}" for label in all_labels],
        columns=[f"pred_{label}" for label in all_labels],
    )

    prediction_columns = [
        column
        for column in ["sample_id", "pdb_id", "protein_name", "split_group", "split_group_source", "first_level_ec_numbers", "graph_path"]
        if column in metadata_for_split.columns
    ]
    predictions = metadata_for_split.iloc[fold.test_idx][prediction_columns].copy()
    predictions.insert(0, "fold_id", fold.fold_id)
    predictions["y_true"] = fold.y_test
    predictions["y_pred"] = y_pred
    predictions["correct"] = predictions["y_true"] == predictions["y_pred"]

    persisted_results = persist_graph_estimator_results(
        results_root=RESULTS_ROOT,
        task=TASK,
        status_path=STATUS_PATH,
        predictions=predictions,
        metrics_df=metrics_df,
        report_df=report_df,
        summary_report_df=summary_report_df,
        confusion_matrix_df=confusion_matrix_df,
        graphs=graphs,
        graphs_for_split=graphs_for_split,
        X_train=fold.X_train,
        X_test=fold.X_test,
        y_train=fold.y_train,
        y_test=fold.y_test,
        test_size=1 / CV_N_SPLITS,
        random_state=RANDOM_STATE,
        split_strategy=split_strategy,
        split_group_column=SPLIT_GROUP_COLUMN,
        split_group_fallback_columns=SPLIT_GROUP_FALLBACK_COLUMNS,
        group_counts=group_counts,
        train_groups=fold.train_groups,
        test_groups=fold.test_groups,
        group_overlap=fold.group_overlap,
        stratified=stratified,
        all_labels=all_labels,
        class_counts=class_counts,
        split_counts=split_counts,
        missing_from_train=fold.missing_from_train,
        missing_from_test=fold.missing_from_test,
        valid_held_out_evaluation=fold.valid_held_out_evaluation,
        min_samples_per_class_for_evaluation=MIN_SAMPLES_PER_CLASS_FOR_EVALUATION,
        filter_under_supported_classes=FILTER_UNDER_SUPPORTED_CLASSES,
        transformer_kind=f"cv{CV_N_SPLITS}_fold{fold.fold_id}_{TRANSFORMER_KIND}",
        node_label_alphabet=NODE_LABEL_ALPHABET,
        transformer=fold_model.transformer,
        base_estimator_kind=BASE_ESTIMATOR_KIND,
        model=fold_model,
        run_timestamp=cv_run_timestamp,
    )

    fold_results.append(
        {
            "fold_id": fold.fold_id,
            "model": fold_model,
            "y_pred": y_pred,
            "predictions": predictions,
            "metrics_df": metrics_df,
            "report_df": report_df,
            "summary_report_df": summary_report_df,
            "confusion_matrix_df": confusion_matrix_df,
            "persisted_results": persisted_results,
        }
    )

len(fold_results)


Fitting fold 0: train=53424 test=13157


/tmp/ipykernel_2934910/2845479663.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  metrics_df.loc["valid_held_out_evaluation", "value"] = fold.valid_held_out_evaluation


Fitting fold 1: train=53425 test=13156


/home/fabrizio/.venvs/py313/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/tmp/ipykernel_2934910/2845479663.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  metrics_df.loc["valid_held_out_evaluation", "value"] = fold.valid_held_out_evaluation


Fitting fold 2: train=52627 test=13954


/tmp/ipykernel_2934910/2845479663.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  metrics_df.loc["valid_held_out_evaluation", "value"] = fold.valid_held_out_evaluation


Fitting fold 3: train=53424 test=13157


/tmp/ipykernel_2934910/2845479663.py:47: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  metrics_df.loc["valid_held_out_evaluation", "value"] = fold.valid_held_out_evaluation


Fitting fold 4: train=53424 test=13157


## 6. Cross-Validation Metrics


In [ ]:
fold_metric_rows = []
for result in fold_results:
    fold_id = result["fold_id"]
    for metric, row in result["summary_report_df"].iterrows():
        fold_metric_rows.append(
            {
                "fold_id": fold_id,
                "metric": metric,
                "value": row["value"],
                "support": row["support"],
            }
        )

fold_metrics_df = pd.DataFrame(fold_metric_rows)
cv_metric_summary = (
    fold_metrics_df.groupby("metric")["value"]
    .agg(["mean", "std", "min", "max"])
    .sort_index()
)

display(fold_metrics_df.pivot(index="fold_id", columns="metric", values="value"))
cv_metric_summary


In [ ]:
report_labels = all_labels
per_class_reports = []
for result in fold_results:
    report_df = result["report_df"].copy()
    report_df.insert(0, "fold_id", result["fold_id"])
    report_df.insert(1, "label", report_df.index)
    per_class_reports.append(report_df.reset_index(drop=True))

per_class_report_df = pd.concat(per_class_reports, ignore_index=True)
per_class_report_summary = (
    per_class_report_df.groupby("label")[["precision", "recall", "f1-score", "support"]]
    .agg({"precision": ["mean", "std"], "recall": ["mean", "std"], "f1-score": ["mean", "std"], "support": "sum"})
)

display(per_class_report_df)
per_class_report_summary


In [ ]:
confusion_matrix_sum = fold_results[0]["confusion_matrix_df"].copy()
for result in fold_results[1:]:
    confusion_matrix_sum = confusion_matrix_sum.add(result["confusion_matrix_df"], fill_value=0)
confusion_matrix_sum = confusion_matrix_sum.astype(int)

cm = confusion_matrix_sum.to_numpy()
cm_row_totals = cm.sum(axis=1, keepdims=True)
cm_fractions = cm / cm_row_totals.clip(min=1)
scale = 0.5
fig, ax = plt.subplots(figsize=(scale * len(report_labels) + 3, scale * len(report_labels) + 2), dpi=130)
im = ax.imshow(cm_fractions, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(report_labels)), labels=[str(label) for label in report_labels], rotation=45, ha="right")
ax.set_yticks(range(len(report_labels)), labels=[str(label) for label in report_labels])
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Cross-Validation Confusion Matrix: count and true-class fraction")

for row_idx in range(cm.shape[0]):
    for col_idx in range(cm.shape[1]):
        fraction = cm_fractions[row_idx, col_idx]
        text_color = "white" if fraction >= 0.55 else "black"
        ax.text(
            col_idx,
            row_idx,
            f"{fraction:.1%}\n[#{cm[row_idx, col_idx]}]",
            ha="center",
            va="center",
            color=text_color,
            fontsize=9,
        )

fig.colorbar(im, ax=ax, label="fraction of true class")
fig.tight_layout()
plt.show()

confusion_matrix_sum


## 7. Inspect Cross-Validation Predictions


In [ ]:
all_predictions = pd.concat([result["predictions"] for result in fold_results], ignore_index=True)
all_predictions.sort_values(["correct", "fold_id", "sample_id"]).head(30)


## 8. Persisted Fold Results


In [ ]:
saved_files = pd.concat(
    [
        result["persisted_results"].to_frame().assign(fold_id=result["fold_id"], run_name=result["persisted_results"].run_name)
        for result in fold_results
    ],
    ignore_index=True,
)
saved_files
